# ☀️ Java: Tmax_annual_p95 × LCZ × Canopy

Minimal analysis to relate **projected heat hazard** (Tmax annual 95th percentile) to **urban form (LCZ)** and **canopy** for **Java Island**.

Principles:
- Use **Tmax_annual_p95** hazard (baseline vs future) from a regional climate model or ensemble.
- Create **LCZ-based modifiers** from historical thermal contrasts (LST–LCZ), then adjust coarse Tmax to 100 m — *relative* downscaling, no subjective weights.
- Quantify **relationships**, not composite scores: (i) per-LCZ Tmax, (ii) canopy–Tmax association, (iii) canopy benefit per LCZ.

👉 Edit the file paths below, then run sequentially.

In [ ]:
# !pip install geopandas rasterio rioxarray xarray numpy pandas shapely pyproj matplotlib scipy

In [ ]:
from pathlib import Path
DATA = Path("./data")        # put rasters here
OUT  = Path("./outputs"); OUT.mkdir(parents=True, exist_ok=True)

paths = {
  # --- REQUIRED INPUTS ---
  # Java boundary polygon for clipping, CRS: any; we'll handle reprojection
  "boundary_java": DATA / "boundary/java_boundary.gpkg",
  # LCZ raster (100 m); LCZ 1-10 = built, 11-17 = natural A..G
  "lcz": DATA / "lcz/lcz_global_java.tif",
  # Canopy Height Map aggregated to ~100 m or native 1 m (we will aggregate)
  "canopy": DATA / "canopy/java_canopy_height.tif",
  # Land Surface Temperature composite (historical warm season) for LCZ differentials
  "lst_hist": DATA / "lst/Java_LST_day_historical.tif",
  # Climate hazard: Tmax annual 95th percentile, baseline and future, coarse grid (e.g., 5–25 km)
  "tmax95_baseline": DATA / "climate/Tmax95_baseline.tif",
  "tmax95_future":   DATA / "climate/Tmax95_future.tif",
}

params = {
  "crs_work": "EPSG:23845",   # Java TM-3 as an example; any metric CRS works
  "grid_res_m": 100,          # master grid
  "canopy_min_height_m": 3,   # canopy presence threshold
}
paths

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.enums import Resampling
from shapely.geometry import box
import matplotlib.pyplot as plt
from scipy.stats import linregress

def reproject_match(src_da: xr.DataArray, ref_da: xr.DataArray, resampling=Resampling.nearest):
    return src_da.rio.reproject_match(ref_da, resampling=resampling)

def make_master_grid(boundary_gdf, res_m=100, crs="EPSG:23845"):
    b = boundary_gdf.to_crs(crs).total_bounds
    xmin, ymin, xmax, ymax = b
    nx = int(np.ceil((xmax - xmin) / res_m))
    ny = int(np.ceil((ymax - ymin) / res_m))
    transform = rasterio.transform.from_origin(xmin, ymax, res_m, res_m)
    width, height = nx, ny
    # create an empty xarray as reference grid
    data = np.zeros((ny, nx), dtype=np.float32)
    da = xr.DataArray(data, dims=("y","x"))
    da = da.rio.write_transform(transform)
    da = da.rio.write_crs(crs)
    return da

def norm01(da):
    vmin = np.nanpercentile(da.values, 2)
    vmax = np.nanpercentile(da.values, 98)
    out = (da - vmin) / (vmax - vmin + 1e-9)
    return out.clip(0, 1)


In [ ]:
# ---- Load boundary & create 100 m master grid ----
java = gpd.read_file(paths["boundary_java"])  # provide any Java polygon
ref = make_master_grid(java, res_m=params["grid_res_m"], crs=params["crs_work"])  # reference grid
ref

In [ ]:
# ---- Load rasters ----
lcz = rxr.open_rasterio(paths["lcz"], masked=True).squeeze()
canopy = rxr.open_rasterio(paths["canopy"], masked=True).squeeze()
lst_hist = rxr.open_rasterio(paths["lst_hist"], masked=True).squeeze()
tmax_b = rxr.open_rasterio(paths["tmax95_baseline"], masked=True).squeeze()
tmax_f = rxr.open_rasterio(paths["tmax95_future"], masked=True).squeeze()

# Clip/align to boundary and 100 m grid
for name, da in [("lcz", lcz), ("canopy", canopy), ("lst_hist", lst_hist)]:
    locals()[name] = reproject_match(da, ref, Resampling.nearest)
tmax_b = reproject_match(tmax_b, ref, Resampling.bilinear)
tmax_f = reproject_match(tmax_f, ref, Resampling.bilinear)
print("All rasters aligned to 100 m grid.")

In [ ]:
# ---- Derive LCZ urban mask and canopy fraction ----
built = ((lcz>=1) & (lcz<=10)).astype("float32")
natural = ((lcz>=11) & (lcz<=17)).astype("float32")

canopy_presence = (canopy >= params["canopy_min_height_m"]).astype("float32")
res_x, res_y = map(abs, ref.rio.resolution())
px_area = res_x * res_y
canopy_area_m2 = canopy_presence * px_area
canopy_frac = canopy_area_m2 / px_area  # equals presence here; kept for clarity

# Save quicklooks
for title, da in [("LCZ", lcz), ("Canopy height", canopy), ("Canopy frac", canopy_frac)]:
    fig, ax = plt.subplots(figsize=(6,6))
    im = ax.imshow(da, origin="upper")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.7)
    plt.show()

## LCZ thermal modifiers from historical LST
Compute mean LST per LCZ, subtract a reference (e.g., LCZ D = low plants, or median of natural LCZs), then use that contrast as an **additive modifier** to distribute coarse Tmax at 100 m.

We also apply an attenuation factor `alpha` (e.g., 0.8) because LST contrasts are typically larger than near-surface air temperature contrasts.

In [ ]:
alpha = 0.8  # attenuation factor from LST contrasts to air-temp contrasts (sensitivity-test later)

classes = [int(c) for c in np.unique(lcz.values[~np.isnan(lcz.values)])]
means = {}
for c in classes:
    sel = xr.where(lcz==c, 1.0, np.nan)
    means[c] = float(np.nanmean((lst_hist * sel).values))
ref_classes = [11,12,13,14]  # natural LCZ A-D if available
ref_vals = [means[c] for c in ref_classes if c in means]
ref_val = float(np.nanmedian(ref_vals)) if len(ref_vals)>0 else float(np.nanmedian(list(means.values())))

lcz_mod = xr.zeros_like(lcz, dtype="float32")
for c, m in means.items():
    delta = (m - ref_val) * alpha
    lcz_mod = xr.where(lcz==c, delta, lcz_mod)

fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(lcz_mod, origin="upper")
ax.set_title("LCZ thermal modifier (°C)")
ax.axis("off")
fig.colorbar(im, ax=ax, shrink=0.7)
plt.show()

## Downscale Tmax_annual_p95
We distribute coarse-grid **Tmax95** to 100 m using LCZ modifiers:

```
Tmax95_100m = Tmax95_coarse_regridded + lcz_mod
```

We compute this for **baseline** and **future**, then take Δ (change).

In [ ]:
tmax95_base_100 = tmax_b + lcz_mod
tmax95_fut_100  = tmax_f + lcz_mod
tmax95_delta    = tmax95_fut_100 - tmax95_base_100

for title, da in [("Tmax95 baseline (°C)", tmax95_base_100), ("Tmax95 future (°C)", tmax95_fut_100), ("ΔTmax95 (°C)", tmax95_delta)]:
    fig, ax = plt.subplots(figsize=(6,6))
    im = ax.imshow(da, origin="upper")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.7)
    plt.show()

## Relationship analysis (no subjective weights)
- **Per-LCZ statistics** of Tmax95 baseline/future and Δ.
- **Canopy–Tmax95 association**: bin or regress Tmax95 vs canopy fraction, globally and **within each LCZ**.
- Report **slopes** (°C per 10% canopy) and R², plus a simple partial view: canopy effect at fixed LCZ.

In [ ]:
# ---- Per-LCZ stats ----
records = []
for c in sorted(classes):
    sel = xr.where(lcz==c, 1.0, np.nan)
    def stat(da):
        v = (da * sel).values
        return float(np.nanmean(v))
    rec = {
        "LCZ": int(c),
        "Tmax95_base": stat(tmax95_base_100),
        "Tmax95_fut":  stat(tmax95_fut_100),
        "Tmax95_delta": stat(tmax95_delta),
        "Canopy_frac": float(np.nanmean((canopy_frac * sel).values)),
    }
    records.append(rec)

df_lcz = pd.DataFrame.from_records(records)
df_lcz.to_csv(OUT / "java_lcz_tmax95_summary.csv", index=False)
df_lcz.head()

In [ ]:
# ---- Canopy–Tmax regression (global and per LCZ) ----
def fit_slope(x, y):
    m = ~np.isnan(x) & ~np.isnan(y)
    if m.sum() < 50:
        return np.nan, np.nan
    r = linregress(x[m], y[m])
    return r.slope, r.rvalue**2

x = canopy_frac.values.flatten()
y = tmax95_base_100.values.flatten()
slope_g, r2_g = fit_slope(x, y)

rows = [{"LCZ":"ALL", "slope_C_per_canopyfrac":slope_g, "R2":r2_g}]
for c in sorted(classes):
    mask = (lcz.values.flatten()==c)
    slope, r2 = fit_slope(x[mask], y[mask])
    rows.append({"LCZ":int(c), "slope_C_per_canopyfrac":slope, "R2":r2})

df_slopes = pd.DataFrame(rows)
df_slopes["slope_C_per_10pct_canopy"] = df_slopes["slope_C_per_canopyfrac"] * 0.10
df_slopes.to_csv(OUT / "java_canopy_tmax_slopes.csv", index=False)
df_slopes.head()

In [ ]:
# ---- Simple scatter (downsampled for speed) ----
rng = np.random.default_rng(42)
idx = rng.choice(x.size, size=min(100000, x.size), replace=False)
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(x[idx], y[idx], s=1, alpha=0.2)
ax.set_xlabel("Canopy fraction (0–1)")
ax.set_ylabel("Tmax95 baseline (°C)")
ax.set_title("Canopy vs Tmax95 (sampled)")
plt.show()

In [ ]:
# ---- Save GeoTIFFs ----
tmax95_base_100.rio.to_raster(OUT / "java_tmax95_baseline_100m.tif", compress="lzw")
tmax95_fut_100.rio.to_raster(OUT / "java_tmax95_future_100m.tif", compress="lzw")
tmax95_delta.rio.to_raster(OUT / "java_tmax95_delta_100m.tif", compress="lzw")
lcz.rio.to_raster(OUT / "java_lcz_100m.tif", compress="lzw")
canopy_frac.rio.to_raster(OUT / "java_canopy_frac_100m.tif", compress="lzw")
print("Saved rasters & CSVs in:", OUT)